# 02 — Prepare filtered BIRD SFT v2 data

This notebook prepares the official 6,601-example filtered BIRD training set,
uses a database-level 90/10 split, applies deterministic 50% Evidence dropout
to training prompts, and builds balanced held-out validation prompts in both
Evidence conditions. Its prompt is intentionally identical to the four-way
Mini-Dev evaluator.


In [1]:
%pip install -q -U "datasets"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 51.5 MB/s eta 0:00:00


In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
from pathlib import Path
import shutil
import subprocess
import zipfile

PROJECT_DIR = Path(
    "/content/drive/MyDrive/bird-text2sql-sft"
)

DATA_DIR = PROJECT_DIR / "data"
ZIP_PATH = DATA_DIR / "bird_train.zip"

LOCAL_ZIP = Path("/content/bird_train.zip")
LOCAL_DATA = Path("/content/bird_data")

if not ZIP_PATH.exists():
    raise FileNotFoundError(
        f"The dataset archive was not found in Google Drive: {ZIP_PATH}"
    )

if not zipfile.is_zipfile(ZIP_PATH):
    raise RuntimeError(
        f"The archive in Google Drive is not a valid ZIP file: {ZIP_PATH}"
    )

print("Dataset archive:", ZIP_PATH)

Dataset archive: /content/drive/MyDrive/bird-text2sql-sft/data/bird_train.zip


In [4]:
if LOCAL_ZIP.exists() and zipfile.is_zipfile(LOCAL_ZIP):
    print("A valid local archive already exists. Skipping the copy step.")
else:
    print("Copying the archive from Google Drive to Colab...")
    shutil.copy2(ZIP_PATH, LOCAL_ZIP)

if not zipfile.is_zipfile(LOCAL_ZIP):
    raise RuntimeError(
        f"The copied local archive is not a valid ZIP file: {LOCAL_ZIP}"
    )

print("Local archive:", LOCAL_ZIP)

Copying the archive from Google Drive to Colab...
Local archive: /content/bird_train.zip


In [5]:
LOCAL_DATA.mkdir(parents=True, exist_ok=True)

nested_database_archives = sorted(
    LOCAL_DATA.rglob("train_databases.zip")
)

if nested_database_archives:
    print("The outer archive has already been extracted.")
else:
    print("Extracting the outer archive...")

    subprocess.run(
        [
            "unzip",
            "-q",
            "-o",
            str(LOCAL_ZIP),
            "-d",
            str(LOCAL_DATA),
        ],
        check=True,
    )

    nested_database_archives = sorted(
        LOCAL_DATA.rglob("train_databases.zip")
    )

if len(nested_database_archives) != 1:
    raise RuntimeError(
        "Expected exactly one train_databases.zip file, "
        f"but found {len(nested_database_archives)}."
    )

INNER_ZIP = nested_database_archives[0]

print("Nested database archive:", INNER_ZIP)

Extracting the outer archive...
Nested database archive: /content/bird_data/train/train_databases.zip


In [6]:
TRAIN_DIR = INNER_ZIP.parent
DATABASE_DIR = TRAIN_DIR / "train_databases"

existing_databases = sorted(
    DATABASE_DIR.rglob("*.sqlite")
)

if existing_databases:
    print(
        f"Found {len(existing_databases)} existing SQLite databases. "
        "Skipping the inner extraction step."
    )
else:
    print("Extracting the database archive...")

    subprocess.run(
        [
            "unzip",
            "-q",
            "-o",
            str(INNER_ZIP),
            "-d",
            str(TRAIN_DIR),
        ],
        check=True,
    )

    existing_databases = sorted(
        DATABASE_DIR.rglob("*.sqlite")
    )

    if not existing_databases:
        raise RuntimeError(
            "The database archive was extracted, "
            "but no SQLite databases were found."
        )

    print(
        f"Database extraction completed. "
        f"Found {len(existing_databases)} SQLite databases."
    )

Extracting the database archive...
Database extraction completed. Found 69 SQLite databases.


In [7]:
database_files = sorted(
    LOCAL_DATA.rglob("*.sqlite")
)

if not database_files:
    raise RuntimeError(
        "No SQLite databases were found under the local data directory."
    )

print("SQLite database count:", len(database_files))

for path in database_files[:10]:
    print(path)

SQLite database count: 138
/content/bird_data/train/__MACOSX/train_databases/address/._address.sqlite
/content/bird_data/train/__MACOSX/train_databases/airline/._airline.sqlite
/content/bird_data/train/__MACOSX/train_databases/app_store/._app_store.sqlite
/content/bird_data/train/__MACOSX/train_databases/authors/._authors.sqlite
/content/bird_data/train/__MACOSX/train_databases/beer_factory/._beer_factory.sqlite
/content/bird_data/train/__MACOSX/train_databases/bike_share_1/._bike_share_1.sqlite
/content/bird_data/train/__MACOSX/train_databases/book_publishing_company/._book_publishing_company.sqlite
/content/bird_data/train/__MACOSX/train_databases/books/._books.sqlite
/content/bird_data/train/__MACOSX/train_databases/car_retails/._car_retails.sqlite
/content/bird_data/train/__MACOSX/train_databases/cars/._cars.sqlite


In [8]:
import json

from datasets import load_dataset


FILTERED_DATASET_ID = "birdsql/bird23-train-filtered"
FILTERED_DATASET_REVISION = "main"
HF_CACHE_DIR = DATA_DIR / "hf_cache"

filtered_train_dataset = load_dataset(
    FILTERED_DATASET_ID,
    split="train",
    revision=FILTERED_DATASET_REVISION,
    cache_dir=str(HF_CACHE_DIR),
)

train_records = [dict(record) for record in filtered_train_dataset]
dataset_fingerprint = filtered_train_dataset._fingerprint

required_fields = {"db_id", "question", "evidence", "SQL"}
missing_fields = required_fields - set(train_records[0])

if missing_fields:
    raise RuntimeError(
        f"Filtered dataset is missing required fields: {sorted(missing_fields)}"
    )

if len(train_records) != 6601:
    raise RuntimeError(
        "Expected 6,601 records from bird23-train-filtered, "
        f"but found {len(train_records)}. Check the dataset revision."
    )

print("Dataset:", FILTERED_DATASET_ID)
print("Revision:", FILTERED_DATASET_REVISION)
print("Dataset fingerprint:", dataset_fingerprint)
print("Filtered training sample count:", len(train_records))
print("Available fields:", train_records[0].keys())
print("First sample:")
print(json.dumps(train_records[0], indent=2, ensure_ascii=False))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/4.42k [00:00<?, ?B/s]

train-00000-of-00001.jsonl:   0%|          | 0.00/2.65M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/6601 [00:00<?, ? examples/s]

Dataset: birdsql/bird23-train-filtered
Revision: main
Dataset fingerprint: 3893d6a6c4df0159
Filtered training sample count: 6601
Available fields: dict_keys(['db_id', 'question', 'evidence', 'SQL'])
First sample:
{
  "db_id": "movie_platform",
  "question": "Who is the director of the movie Sex, Drink and Bloodshed?",
  "evidence": "Sex, Drink and Bloodshed refers to movie title = 'Sex, Drink and Bloodshed';",
  "SQL": "SELECT director_name FROM movies WHERE movie_title = 'Sex, Drink and Bloodshed'"
}


In [9]:
database_path_by_id = {}

for db_path in database_files:
    db_id = db_path.stem

    if db_id in database_path_by_id:
        raise RuntimeError(
            f"Duplicate database ID detected: {db_id}"
        )

    database_path_by_id[db_id] = db_path

print("Database mapping count:", len(database_path_by_id))

sample_db_ids = {
    record["db_id"]
    for record in train_records
}

missing_db_ids = sorted(
    sample_db_ids - set(database_path_by_id)
)

print("Unique database IDs in training data:", len(sample_db_ids))
print("Missing database IDs:", len(missing_db_ids))

if missing_db_ids:
    print("First missing database IDs:", missing_db_ids[:10])
    raise RuntimeError(
        "Some training samples do not have matching SQLite databases."
    )

print("All training samples have matching databases.")

Database mapping count: 138
Unique database IDs in training data: 69
Missing database IDs: 0
All training samples have matching databases.


In [10]:
import sqlite3

def extract_schema(db_path):
    connection = sqlite3.connect(
        f"file:{db_path.resolve()}?mode=ro",
        uri=True,
    )

    try:
        rows = connection.execute(
            """
            SELECT name, sql
            FROM sqlite_master
            WHERE type = 'table'
              AND name NOT LIKE 'sqlite_%'
              AND sql IS NOT NULL
            ORDER BY name
            """
        ).fetchall()

        schema_parts = []

        for table_name, create_statement in rows:
            schema_parts.append(create_statement.strip() + ";")

        return "\n\n".join(schema_parts)

    finally:
        connection.close()

In [11]:
schema_by_db_id = {}

for db_id in sorted(sample_db_ids):
    db_path = database_path_by_id[db_id]
    schema_by_db_id[db_id] = extract_schema(db_path)

print("Extracted schema count:", len(schema_by_db_id))

Extracted schema count: 69


In [12]:
PROMPT_VERSION = "bird_schema_evidence_ablation_chat_v2"

SYSTEM_PROMPT = (
    "You are an expert text-to-SQL assistant. Given a SQLite database schema, "
    "external knowledge, and a question, write one correct SQLite query. "
    "Return only the SQL query, without Markdown fences or explanation."
)


In [13]:
def normalized_evidence(record):
    return str(record.get("evidence") or "").strip()


def build_user_prompt(record, schema_text, use_evidence):
    evidence = normalized_evidence(record)
    evidence_block = evidence if (use_evidence and evidence) else "None"

    return f'''Database ID:
{record["db_id"]}

SQLite schema:
{schema_text}

External knowledge:
{evidence_block}

Question:
{record["question"]}

Write exactly one executable SQLite query that answers the question. Return SQL only.'''


In [14]:
def build_sft_record(source_index, record, use_evidence):
    db_id = record["db_id"]
    schema_text = schema_by_db_id[db_id]

    return {
        "source_index": int(source_index),
        "db_id": db_id,
        "evidence": normalized_evidence(record),
        "use_evidence": bool(use_evidence),
        "prompt_version": PROMPT_VERSION,
        "messages": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": build_user_prompt(
                    record,
                    schema_text,
                    use_evidence=use_evidence,
                ),
            },
            {
                "role": "assistant",
                "content": record["SQL"].strip(),
            },
        ],
    }


In [15]:
sample_no_evidence = build_sft_record(
    0,
    train_records[0],
    use_evidence=False,
)

sample_with_evidence = build_sft_record(
    0,
    train_records[0],
    use_evidence=True,
)

print("NO-EVIDENCE TRAINING PROMPT:")
print(json.dumps(sample_no_evidence["messages"][:-1], indent=2, ensure_ascii=False))
print()
print("WITH-EVIDENCE TRAINING PROMPT:")
print(json.dumps(sample_with_evidence["messages"][:-1], indent=2, ensure_ascii=False))


NO-EVIDENCE TRAINING PROMPT:
[
  {
    "role": "system",
    "content": "You are an expert text-to-SQL assistant. Given a SQLite database schema, external knowledge, and a question, write one correct SQLite query. Return only the SQL query, without Markdown fences or explanation."
  },
  {
    "role": "user",
    "content": "Database ID:\nmovie_platform\n\nSQLite schema:\nCREATE TABLE \"lists\"\n(\n    user_id                     INTEGER\n        references lists_users (user_id),\n    list_id                     INTEGER not null\n        primary key,\n    list_title                  TEXT,\n    list_movie_number           INTEGER,\n    list_update_timestamp_utc   TEXT,\n    list_creation_timestamp_utc TEXT,\n    list_followers              INTEGER,\n    list_url                    TEXT,\n    list_comments               INTEGER,\n    list_description            TEXT,\n    list_cover_image_url        TEXT,\n    list_first_image_url        TEXT,\n    list_second_image_url       TEXT,\n    

In [16]:
import random
from collections import defaultdict


SPLIT_SEED = 42
EVIDENCE_DROPOUT_RATE = 0.50

records_by_db_id = defaultdict(list)

for source_index, record in enumerate(train_records):
    records_by_db_id[record["db_id"]].append((source_index, record))

all_db_ids = sorted(records_by_db_id)
split_rng = random.Random(SPLIT_SEED)
split_rng.shuffle(all_db_ids)

validation_db_count = max(1, round(len(all_db_ids) * 0.10))
validation_db_ids = set(all_db_ids[:validation_db_count])
training_db_ids = set(all_db_ids[validation_db_count:])

training_source_records = [
    pair
    for db_id in sorted(training_db_ids)
    for pair in records_by_db_id[db_id]
]
validation_source_records = [
    pair
    for db_id in sorted(validation_db_ids)
    for pair in records_by_db_id[db_id]
]

# Apply deterministic 50% evidence dropout only after the database-level split.
# Each training question appears once, in exactly one evidence condition.
eligible_training_positions = [
    position
    for position, (_, record) in enumerate(training_source_records)
    if normalized_evidence(record)
]
dropout_rng = random.Random(SPLIT_SEED)
dropout_rng.shuffle(eligible_training_positions)

target_with_evidence = min(
    len(eligible_training_positions),
    round(len(training_source_records) * (1.0 - EVIDENCE_DROPOUT_RATE)),
)
with_evidence_positions = set(
    eligible_training_positions[:target_with_evidence]
)

train_sft_records = [
    build_sft_record(
        source_index,
        record,
        use_evidence=(position in with_evidence_positions),
    )
    for position, (source_index, record) in enumerate(training_source_records)
]

# Validation is explicitly balanced: every example with evidence appears in
# both prompt conditions. This makes validation loss reflect both capabilities.
validation_sft_records = []
for source_index, record in validation_source_records:
    conditions = [False, True] if normalized_evidence(record) else [False]
    for use_evidence in conditions:
        validation_sft_records.append(
            build_sft_record(source_index, record, use_evidence)
        )

all_prepared_records = train_sft_records + validation_sft_records

print("Filtered source records:", len(train_records))
print("Training source records:", len(training_source_records))
print("Validation source records:", len(validation_source_records))
print("Prepared training prompts:", len(train_sft_records))
print("Prepared validation prompts:", len(validation_sft_records))


Filtered source records: 6601
Training source records: 5825
Validation source records: 776
Prepared training prompts: 5825
Prepared validation prompts: 1485


In [17]:
EXPECTED_ROLES = ["system", "user", "assistant"]

validation_errors = []

for index, record in enumerate(all_prepared_records):
    messages = record.get("messages", [])
    roles = [message.get("role") for message in messages]

    if not isinstance(record.get("source_index"), int):
        validation_errors.append((index, "Invalid source_index"))

    if not record.get("db_id"):
        validation_errors.append((index, "Missing db_id"))

    if not isinstance(record.get("use_evidence"), bool):
        validation_errors.append((index, "Invalid use_evidence"))

    if record.get("prompt_version") != PROMPT_VERSION:
        validation_errors.append((index, "Prompt version mismatch"))

    if roles != EXPECTED_ROLES:
        validation_errors.append((index, f"Invalid roles: {roles}"))
        continue

    for message in messages:
        content = message.get("content")
        if not isinstance(content, str) or not content.strip():
            validation_errors.append(
                (index, f"Empty or invalid {message.get('role')} message")
            )

print("Prepared SFT prompts:", len(all_prepared_records))
print("Validation error count:", len(validation_errors))

for error in validation_errors[:10]:
    print(error)

if validation_errors:
    raise RuntimeError("The generated SFT dataset contains invalid records.")

print("All SFT records passed structural validation.")


Prepared SFT prompts: 7310
Validation error count: 0
All SFT records passed structural validation.


In [18]:
from collections import Counter


sample_keys = [
    (
        record["db_id"],
        record["messages"][1]["content"],
        record["messages"][2]["content"],
    )
    for record in all_prepared_records
]

sample_key_counts = Counter(sample_keys)
duplicate_count = sum(
    count - 1
    for count in sample_key_counts.values()
    if count > 1
)

print("Exact duplicate prompt/completion count:", duplicate_count)

if duplicate_count:
    duplicate_examples = [
        (key, count)
        for key, count in sample_key_counts.items()
        if count > 1
    ]
    for key, count in duplicate_examples[:5]:
        print("Occurrence count:", count)
        print("Database ID:", key[0])
        print("SQL:", key[2])
        print("-" * 80)


Exact duplicate prompt/completion count: 0


In [19]:
def condition_counts(records):
    return Counter(
        "with_evidence" if record["use_evidence"] else "no_evidence"
        for record in records
    )


print("Training condition counts:", dict(condition_counts(train_sft_records)))
print("Validation condition counts:", dict(condition_counts(validation_sft_records)))

train_with_evidence_rate = sum(
    record["use_evidence"] for record in train_sft_records
) / len(train_sft_records)

print(f"Training prompts with evidence: {train_with_evidence_rate:.2%}")
print(f"Training evidence dropout: {1.0 - train_with_evidence_rate:.2%}")

assert abs(train_with_evidence_rate - 0.50) < 0.02


Training condition counts: {'with_evidence': 2912, 'no_evidence': 2913}
Validation condition counts: {'no_evidence': 776, 'with_evidence': 709}
Training prompts with evidence: 49.99%
Training evidence dropout: 50.01%


In [20]:
user_prompt_lengths = [
    len(record["messages"][1]["content"])
    for record in all_prepared_records
]

sql_lengths = [
    len(record["messages"][2]["content"])
    for record in all_prepared_records
]


def print_length_statistics(name, lengths):
    values = sorted(lengths)
    print(name)
    print("Minimum:", min(values))
    print("Median:", values[len(values) // 2])
    print("95th percentile:", values[int(len(values) * 0.95)])
    print("Maximum:", max(values))
    print()


print_length_statistics("User prompt character lengths", user_prompt_lengths)
print_length_statistics("SQL character lengths", sql_lengths)


User prompt character lengths
Minimum: 572
Median: 2774
95th percentile: 37269
Maximum: 37705

SQL character lengths
Minimum: 23
Median: 160
95th percentile: 289
Maximum: 804



In [21]:
print("Total database count:", len(all_db_ids))
print("Training database count:", len(training_db_ids))
print("Validation database count:", len(validation_db_ids))
print("Training database IDs:", sorted(training_db_ids))
print("Validation database IDs:", sorted(validation_db_ids))


Total database count: 69
Training database count: 62
Validation database count: 7
Training database IDs: ['address', 'airline', 'app_store', 'authors', 'beer_factory', 'bike_share_1', 'book_publishing_company', 'car_retails', 'cars', 'chicago_crime', 'citeseer', 'codebase_comments', 'coinmarketcap', 'college_completion', 'cookbook', 'craftbeer', 'cs_semester', 'disney', 'donor', 'european_football_1', 'food_inspection_2', 'genes', 'hockey', 'human_resources', 'ice_hockey_draft', 'image_and_language', 'language_corpus', 'legislator', 'mental_health_survey', 'menu', 'mondial_geo', 'movie', 'movie_platform', 'movielens', 'movies_4', 'music_platform_2', 'music_tracker', 'olympics', 'professional_basketball', 'public_review_platform', 'regional_sales', 'restaurant', 'retail_complains', 'retail_world', 'retails', 'sales_in_weather', 'shakespeare', 'shipping', 'shooting', 'simpson_episodes', 'soccer_2016', 'social_media', 'software_company', 'student_loan', 'synthea', 'talkingdata', 'trains',

In [22]:
database_overlap = training_db_ids & validation_db_ids
training_source_indices = {index for index, _ in training_source_records}
validation_source_indices = {index for index, _ in validation_source_records}

print("Overlapping database count:", len(database_overlap))
print(
    "Overlapping source record count:",
    len(training_source_indices & validation_source_indices),
)

assert not database_overlap
assert not (training_source_indices & validation_source_indices)
assert len(training_source_records) + len(validation_source_records) == len(train_records)
assert len(train_sft_records) == len(training_source_records)

validation_conditions_by_source = defaultdict(set)
for record in validation_sft_records:
    validation_conditions_by_source[record["source_index"]].add(
        record["use_evidence"]
    )

for source_index, record in validation_source_records:
    expected = {False, True} if normalized_evidence(record) else {False}
    assert validation_conditions_by_source[source_index] == expected

print("The database-level split and balanced validation variants are valid.")


Overlapping database count: 0
Overlapping source record count: 0
The database-level split and balanced validation variants are valid.


In [23]:
OUTPUT_DIR = PROJECT_DIR / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_OUTPUT_PATH = OUTPUT_DIR / "train_sft_v2.jsonl"
VALIDATION_OUTPUT_PATH = OUTPUT_DIR / "validation_sft_v2.jsonl"
SPLIT_OUTPUT_PATH = OUTPUT_DIR / "split_metadata_v2.json"


In [24]:
def save_jsonl(records, output_path):
    with open(
        output_path,
        "w",
        encoding="utf-8",
    ) as file:
        for record in records:
            file.write(
                json.dumps(
                    record,
                    ensure_ascii=False,
                )
                + "\n"
            )

In [25]:
save_jsonl(train_sft_records, TRAIN_OUTPUT_PATH)
save_jsonl(validation_sft_records, VALIDATION_OUTPUT_PATH)

split_metadata = {
    "dataset_id": FILTERED_DATASET_ID,
    "dataset_revision": FILTERED_DATASET_REVISION,
    "dataset_fingerprint": dataset_fingerprint,
    "filtered_source_count": len(train_records),
    "random_seed": SPLIT_SEED,
    "split_method": "database_level_90_10",
    "evidence_dropout_rate": EVIDENCE_DROPOUT_RATE,
    "prompt_version": PROMPT_VERSION,
    "system_prompt": SYSTEM_PROMPT,
    "training_database_ids": sorted(training_db_ids),
    "validation_database_ids": sorted(validation_db_ids),
    "training_source_count": len(training_source_records),
    "validation_source_count": len(validation_source_records),
    "training_prompt_count": len(train_sft_records),
    "validation_prompt_count": len(validation_sft_records),
    "training_condition_counts": dict(condition_counts(train_sft_records)),
    "validation_condition_counts": dict(condition_counts(validation_sft_records)),
}

with open(SPLIT_OUTPUT_PATH, "w", encoding="utf-8") as file:
    json.dump(split_metadata, file, indent=2, ensure_ascii=False)

print("Training file:", TRAIN_OUTPUT_PATH)
print("Validation file:", VALIDATION_OUTPUT_PATH)
print("Split metadata:", SPLIT_OUTPUT_PATH)


Training file: /content/drive/MyDrive/bird-text2sql-sft/processed/train_sft_v2.jsonl
Validation file: /content/drive/MyDrive/bird-text2sql-sft/processed/validation_sft_v2.jsonl
Split metadata: /content/drive/MyDrive/bird-text2sql-sft/processed/split_metadata_v2.json


In [26]:
def load_jsonl(input_path):
    records = []

    with open(
        input_path,
        "r",
        encoding="utf-8",
    ) as file:
        for line_number, line in enumerate(file, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as error:
                raise RuntimeError(
                    f"Invalid JSON on line {line_number}: {input_path}"
                ) from error

    return records

In [27]:
loaded_train_records = load_jsonl(TRAIN_OUTPUT_PATH)
loaded_validation_records = load_jsonl(VALIDATION_OUTPUT_PATH)

assert len(loaded_train_records) == len(train_sft_records)
assert len(loaded_validation_records) == len(validation_sft_records)
assert {
    record["prompt_version"] for record in loaded_train_records
} == {PROMPT_VERSION}
assert {
    record["prompt_version"] for record in loaded_validation_records
} == {PROMPT_VERSION}

print("Saved training prompts:", len(loaded_train_records))
print("Saved validation prompts:", len(loaded_validation_records))
print("The v2 JSONL files were saved and reloaded successfully.")


Saved training prompts: 5825
Saved validation prompts: 1485
The v2 JSONL files were saved and reloaded successfully.
